In [ ]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
import json
from datasets import Dataset

DATA_PATH = "/kaggle/input/datasets/stratoshift/datafile/charakasamhita_parallel.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

dataset

In [ ]:
def format_example(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": example["original"]
            },
            {
                "role": "assistant",
                "content": example["sandhi_split"]
            }
        ]
    }

dataset = dataset.map(format_example)
dataset[0]

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,

    r = 16,          # you can try 8 / 16 / 32
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

dataset[0]["text"]

In [ ]:
dataset = dataset.train_test_split(test_size=0.02)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 200,   # increase for full training
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()

Inference

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

messages = [{
    "role": "user",
    "content": "तदुपासनीयम्"
}]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
)

inputs = tokenizer([text], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    temperature=0.7,
    top_p=0.9,
)

print(tokenizer.batch_decode(outputs)[0])

In [ ]:
model.save_pretrained("sandhi_lora_model")
tokenizer.save_pretrained("sandhi_lora_model")

In [ ]:
from unsloth.chat_templates import get_chat_template
import torch

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

def predict(word):
    messages = [{
        "role": "user",
        "content": word
    }]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )

    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.3,   # lower = more accurate for Sandhi
        top_p=0.9,
    )

    decoded = tokenizer.batch_decode(outputs)[0]

    return decoded

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

# ---------------------------
# Load trained model
# ---------------------------
model, tokenizer = FastModel.from_pretrained(
    model_name = "sandhi_lora_model",  # or full path
    max_seq_length = 2048,
    load_in_4bit = True,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# ---------------------------
# Prediction function
# ---------------------------
def predict(word):
    messages = [{
        "role": "user",
        "content": word
    }]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )

    inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.2,
            do_sample=False
        )

    result = tokenizer.batch_decode(outputs)[0]
    return result.split("<start_of_turn>model")[-1].strip()

# ---------------------------
# YOUR INPUT HERE
# ---------------------------
user_input = input("Enter Sanskrit word/line: ")
print("\nINPUT :", user_input)
print("OUTPUT:", predict(user_input))

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

FastModel.for_inference(model)

def predict(text):
    messages = [
        {
            "role": "user",
            "content": text,
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.1,
        top_p=0.95,
        do_sample=False,
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    # Remove prompt portion
    response = response.split("model")[-1].strip()

    return response


while True:
    text = input("Enter Sanskrit word/line: ")

    if text.lower() == "q":
        break

    result = predict(text)

    print("\nINPUT :", text)
    print("OUTPUT:", result)
    print("-" * 60)

In [1]:
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

import torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# -------------------------
# FORCE SINGLE GPU
# -------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = "cuda"

print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# CLEAR CUDA CACHE
# -------------------------

torch.cuda.empty_cache()

# -------------------------
# LOAD MODEL
# -------------------------

model, tokenizer = FastModel.from_pretrained(
    model_name="sandhi_lora_model",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Enable inference optimizations
FastModel.for_inference(model)

# -------------------------
# CHAT TEMPLATE
# -------------------------

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

# -------------------------
# PREDICTION FUNCTION
# -------------------------

def predict(text):

    messages = [
        {
            "role": "user",
            "content": f"Split the Sanskrit sandhi:\n{text}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            temperature=0.0,
            use_cache=True,
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    return decoded


# -------------------------
# TEST
# -------------------------

test_words = [
    "तदुपासनीयम्",
    "रामोऽस्ति",
]

for word in test_words:

    print("\nINPUT :", word)

    try:
        result = predict(word)
        print("OUTPUT:", result)

    except Exception as e:
        print("ERROR:", e)

    print("-" * 50)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 



INPUT : तदुपासनीयम्
OUTPUT: user
Split the Sanskrit sandhi:
तदुपासनीयम्
model
तदुपासनीयम्
--------------------------------------------------

INPUT : रामोऽस्ति
OUTPUT: user
Split the Sanskrit sandhi:
रामोऽस्ति
model
रामः+अस्ति
--------------------------------------------------
